# Problem Statement
1. **Spectra's Objective**: The sports magazine wants to repurpose unused cricket images for social media posts under the #throwback category to boost engagement during off-seasons.

2. **Engagement Strategy**: These posts aim to drive interaction on social media by filling the content gap when there are no live updates to share.

3. **Image Analyser System**: Develop an AG2-based image analyzer that identifies the sport, analyzes team dynamics and winning probabilities, and generates engaging commentary for the images.

1. https://i.ytimg.com/vi/CarUZqvtzeU/maxresdefault.jpg
2. https://i.ytimg.com/vi/SAuwTBsRMcg/maxresdefault.jpg

In [ ]:
#!pip install Pillow

In [ ]:
import os 
import warnings
warnings.filterwarnings("ignore")
import autogen
from autogen import ConversableAgent
from autogen.agentchat.contrib.multimodal_conversable_agent import MultimodalConversableAgent
from IPython.display import Markdown, display, Image

In [ ]:
# load env variables
from dotenv import load_dotenv
load_dotenv('/Users/admin/Desktop/AutoGen/Module 1/.env')

In [ ]:
# configuration for LLM
config_list_1 = {
    "config_list": [{"model": "gpt-4o", "temperature": 0.3}]
}

In [ ]:
# configuration for LLM
config_list_2 = {
    "config_list": [{"model": "gpt-4o", "temperature": 0.5}]
}

In [ ]:
game_analyzer_agent = MultimodalConversableAgent(
    "Game_Analyzer_Agent",
    system_message = """
You are tasked with analyzing a sports image to determine the game being played. 
Identify the type of sport (e.g., football, basketball, soccer,cricket or any other) 
from the image and recognize the teams that are involved. 
Provide basic details such as the names of players and name of the teams and the current 
state of the game (e.g., score, quarter, half-time status, overs reaminnig etc.). 
If there are any visible details that provide additional details about the game, mention them as well. 
""",
    llm_config=config_list_2,
    human_input_mode="NEVER",
)


In [ ]:
probability_agent = ConversableAgent(
    "Probability_Analyzer_Agent",
    system_message = """
You are tasked with calculating the winning probability for each team based on the current score and the score they need to chase. 
Search across the web to collect information about the match and Use the game stats and their historical performences and 
calculate the probability of each team winning based on their performance up until now. Consider factors such as the score gap, time left in the game, 
and give the approximate percentage of each teams for wining the game.
""",
    llm_config= config_list_1,
    human_input_mode='NEVER',
)

In [ ]:
commentory_agent = MultimodalConversableAgent(
    "Commentory_agent",
    system_message = """
You are tasked with delivering a detailed, real-time commentary on the game:
- Take help of the Game Analyzer Agent's findings (game type, score, teams, etc.).
- Use the Probability Agent's calculations to provide insight into the game's likely outcomes.
Focus on creating a seamless and detailed narrative that provides key moments, highlights, and ongoing excitement for fans.
""",
    llm_config= config_list_2,
    human_input_mode="NEVER",
)

In [ ]:
user_proxy = autogen.UserProxyAgent(
    name="User_proxy",
    system_message="""
    You are assisting in analyzing sports images and coordinating among agents to extract relevant game details. If the input contains URLs, fetch and incorporate relevant details.
""",
    human_input_mode="TERMINATE",  # Try between ALWAYS, NEVER, and TERMINATE
    max_consecutive_auto_reply=0,
    code_execution_config={
        "use_docker": False  # Set to True if Docker is available
    },
)

In [ ]:
group_meet = autogen.GroupChat(
    agents=[user_proxy,game_analyzer_agent, probability_agent, commentory_agent],
    messages=[],
    max_round=4,
    speaker_selection_method= "auto",
    select_speaker_auto_llm_config = config_list_1
)
group_manager = autogen.GroupChatManager(
    groupchat=group_meet,
)

In [ ]:
meeting = user_proxy.initiate_chat(
    group_manager,
    message="""Analyze the sports image  and extract the game type or  teams involved or any visible players or score
    <img https://i.ytimg.com/vi/CarUZqvtzeU/maxresdefault.jpg>""",
summary_method="last_msg",
)

In [ ]:
agent_name_to_find = ["Game_Analyzer_Agent", "Probability_Analyzer_Agent", "Commentory_agent"]

display(Image(url="https://i.ytimg.com/vi/CarUZqvtzeU/maxresdefault.jpg", width = 600))
for agent_name in agent_name_to_find:

    # Iterate through the dictionary to find the last conversation for the specified agent
    for agent, interactions in group_manager.chat_messages.items():
        # Filter interactions by agent name
        filtered_interactions = [i for i in interactions if i.get("name") == agent_name]
        if filtered_interactions:
            # Print the last conversation's content
            # print(filtered_interactions[-1]["content"])
            display(Markdown('## ' + agent_name))
            display(Markdown(filtered_interactions[-1]["content"]))
            break
    else:
        print("Agent not found or no interactions available.")


In [ ]:
meeting1 = user_proxy.initiate_chat(
    group_manager,
    message="""Analyze the sports image  and extract the game type or  teams involved or any visible players or score
    <img https://i.ytimg.com/vi/SAuwTBsRMcg/maxresdefault.jpg>.""",
summary_method="last_msg",
)

In [ ]:
agent_name_to_find = ["Game_Analyzer_Agent", "Probability_Analyzer_Agent", "Commentory_agent"]

display(Image(url="https://i.ytimg.com/vi/SAuwTBsRMcg/maxresdefault.jpg", width = 600))

for agent_name in agent_name_to_find:

    # Iterate through the dictionary to find the last conversation for the specified agent
    for agent, interactions in group_manager.chat_messages.items():
        # Filter interactions by agent name
        filtered_interactions = [i for i in interactions if i.get("name") == agent_name]
        if filtered_interactions:
            # Print the last conversation's content
            # print(filtered_interactions[-1]["content"])
            display(Markdown('## ' + agent_name))
            display(Markdown(filtered_interactions[-1]["content"]))
            break
    else:
        print("Agent not found or no interactions available.")
